# Criando Digital Twins para o artigo da matéria de Doutorado do PPCA (Fatores Humanos na Cibersegurança)

Pipeline simples para criar personas organizacionais brasileiras, instanciar cada persona como um Human Digital Twin via LLM local e gerar respostas sintéticas em escala Likert de 1 a 7 para o modelo 7S de Proteção Tecnológica Organizacional.


## Configurações Gerais

Altere as constantes abaixo antes de executar o notebook.


In [2]:
from utils import GerarDadosUtils

# Configurações
SEED = 84
# N_PERSONAS = 3
N_PERSONAS = 1
N_SESSOES_POR_PERSONA = 1
LLM_PROVIDER = "ollama"
OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "gemma4:e2b"
OPENAI_MODEL = "gpt-5.4-nano-2026-03-17"

utils = GerarDadosUtils(
    seed=SEED,
    n_personas=N_PERSONAS,
    n_sessoes_por_persona=N_SESSOES_POR_PERSONA,
    llm_provider=LLM_PROVIDER,
    ollama_host=OLLAMA_HOST,
    ollama_model=OLLAMA_MODEL,
    openai_model=OPENAI_MODEL,
)

## 1. Configuração do LLM (Ollama ou OpenAI)


In [3]:
# Teste rápido da conexão com o LLM
# utils.get_response("Olá, qual é a capital da França?", temperature=0.5)

## 2. Ajustando Dados Reais e Gerando Personas


In [4]:
# Formatar dataset real
df_real = utils.formatar_dataset_real("inputs/questionario_real.csv")
df_real.to_csv("inputs/questionario_real_formatado.csv", index=False)
df_real.head(2)

,SV1,SV2,SV3,SV4,SG1,SG2,SG3,SG4,SU1,SU2,...,TP2,TP3,TP4,idade,sexo,setor_trabalha,porte_org,funcao_cargo,tempo_atuacao_org_anos,ja_recebeu_treinamento
0,6,6,5,4,6,3,3,2,6,7,...,4,3,2,36,Masculino,Pública,Grande (250 ou mais),"Operacional/técnica (analista, desenvolvedor, ...",11,Não
1,7,7,7,7,7,7,7,7,7,7,...,5,6,7,24,Masculino,Privada,Grande (250 ou mais),"Operacional/técnica (analista, desenvolvedor, ...",0,Sim


In [5]:
# Gerar personas e sobrescrever com distribuição real
personas_df = utils.gerar_personas_df()
personas_df = utils.sobrescrever_com_distribuicao_real(personas_df, df_real)
personas_df.head(2)

,persona_id,idade,sexo,setor_trabalha,porte_org,funcao_cargo,tempo_atuacao_org_anos,ja_recebeu_treinamento,pais
0,HDT_001,25,Feminino,Privada,Média (50 a 249),"Operacional/técnica (analista, desenvolvedor, ...",18,Não,Brasil


### 2.1 Estatísticas descritivas das personas


In [6]:
import pandas as pd

label_map = {
    "idade": "Idade",
    "sexo": "Sexo",
    "setor_trabalha": "Setor Econômico",
    "porte_org": "Porte Organizacional",
    "funcao_cargo": "Cargo Hierárquico",
    "tempo_atuacao_org_anos": "Tempo de Atuação na Organização",
    "ja_recebeu_treinamento": "Já recebeu treinamento",
}

bins_idade = [17, 30, 40, 60]
labels_idade = ["18-30", "31-40", "41-60"]
bins_tempo = [-1, 10, 20, 31]
labels_tempo = ["0-10", "11-20", "21-31"]

all_cats = {
    "idade": labels_idade,
    "sexo": utils.SEXO_OPCOES,
    "setor_trabalha": utils.SETOR_TRABALHA_OPCOES,
    "porte_org": utils.PORTE_ORG_OPCOES,
    "funcao_cargo": utils.FUNCAO_CARGOS_OPCOES,
    "tempo_atuacao_org_anos": labels_tempo,
    "ja_recebeu_treinamento": utils.JA_RECEBEU_TREINAMENTO_OPCOES,
}

rows = []
for col, label in label_map.items():
    if col == "idade":
        series = pd.cut(personas_df[col], bins=bins_idade, labels=labels_idade)
    elif col == "tempo_atuacao_org_anos":
        series = pd.cut(personas_df[col], bins=bins_tempo, labels=labels_tempo)
    else:
        series = personas_df[col]
    counts = series.value_counts().reindex(all_cats[col], fill_value=0)
    for val, n in counts.items():
        rows.append(
            {
                "Atributo": label,
                "Categoria": val,
                "n": n,
                "\\%": f"{100 * n // len(personas_df)}\\%",
            }
        )

desc_df = pd.DataFrame(rows)
desc_df

,Atributo,Categoria,n,\%
0,Idade,18-30,1,100\%
1,Idade,31-40,0,0\%
2,Idade,41-60,0,0\%
3,Sexo,Masculino,0,0\%
4,Sexo,Feminino,1,100\%
5,Sexo,Prefiro não informar,0,0\%
6,Setor Econômico,Pública,0,0\%
7,Setor Econômico,Privada,1,100\%
8,Porte Organizacional,Grande (250 ou mais),0,0\%
9,Porte Organizacional,Média (50 a 249),1,100\%


## 3. Questionário do modelo 7S e Proteção Tecnológica


In [7]:
questionario_df = pd.DataFrame(utils.QUESTIONARIO_7S)
questionario_df

,item,dimensao,tipo_variavel,afirmacao
0,SV1,Shared Values,independente,Minha empresa compartilha a importância e o va...
1,SV2,Shared Values,independente,Minha empresa inclui o conceito de proteção te...
2,SV3,Shared Values,independente,Minha organização está ciente da necessidade d...
3,SV4,Shared Values,independente,Minha empresa tem uma cultura organizacional q...
4,SG1,Strategy,independente,Minha empresa possui uma estratégia de proteçã...
5,SG2,Strategy,independente,"Para minha empresa, a proteção tecnológica é u..."
6,SG3,Strategy,independente,Minha empresa possui um plano de proteção tecn...
7,SG4,Strategy,independente,Minha empresa investe um orçamento suficiente ...
8,SU1,Structure,independente,Minha empresa possui uma estrutura organizacio...
9,SU2,Structure,independente,Minha empresa possui um departamento encarrega...


## 4. Instanciação dos Human Digital Twins e geração das respostas


In [8]:
# Calcular distribuição-alvo
target_probs = utils.calcular_target_probs(df_real)
target_probs

{'SV1': {1: 0.0, 2: 0.043, 3: 0.065, 4: 0.087, 5: 0.13, 6: 0.239, 7: 0.435},
 'SV2': {1: 0.0, 2: 0.065, 3: 0.022, 4: 0.109, 5: 0.217, 6: 0.239, 7: 0.348},
 'SV3': {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.065, 5: 0.13, 6: 0.196, 7: 0.609},
 'SV4': {1: 0.022, 2: 0.022, 3: 0.043, 4: 0.152, 5: 0.196, 6: 0.217, 7: 0.348},
 'SG1': {1: 0.0, 2: 0.022, 3: 0.087, 4: 0.109, 5: 0.196, 6: 0.326, 7: 0.261},
 'SG2': {1: 0.0, 2: 0.043, 3: 0.196, 4: 0.087, 5: 0.196, 6: 0.152, 7: 0.326},
 'SG3': {1: 0.0, 2: 0.087, 3: 0.065, 4: 0.196, 5: 0.087, 6: 0.304, 7: 0.261},
 'SG4': {1: 0.022, 2: 0.109, 3: 0.065, 4: 0.239, 5: 0.239, 6: 0.087, 7: 0.239},
 'SU1': {1: 0.043, 2: 0.022, 3: 0.087, 4: 0.065, 5: 0.152, 6: 0.391, 7: 0.239},
 'SU2': {1: 0.065, 2: 0.022, 3: 0.043, 4: 0.043, 5: 0.109, 6: 0.217, 7: 0.5},
 'SU3': {1: 0.043, 2: 0.022, 3: 0.043, 4: 0.065, 5: 0.109, 6: 0.239, 7: 0.478},
 'SU4': {1: 0.043, 2: 0.043, 3: 0.043, 4: 0.13, 5: 0.13, 6: 0.348, 7: 0.261},
 'SM1': {1: 0.0, 2: 0.043, 3: 0.043, 4: 0.152, 5: 0.174, 6: 0.

## 5. Execução das sessões independentes


In [9]:
from pathlib import Path
from tqdm.notebook import tqdm

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

personas = personas_df.to_dict(orient="records")

resultados = []
for sessao in range(1, N_SESSOES_POR_PERSONA + 1):
    for persona in tqdm(personas, desc="Gerando respostas"):
        try:
            linha = utils.responder_persona(persona, target_probs, sessao=sessao)
            linha["sessao"] = sessao
            resultados.append(linha)
        except Exception as e:
            print(f"Erro em {persona['persona_id']} sessão {sessao}: {e}")

df_respostas = pd.DataFrame(resultados)
df_respostas.head(2)

Gerando respostas:   0%|          | 0/1 [00:00<?, ?it/s]

Erro em HDT_001 sessão 1: model 'gemma4:e2b' not found (status code: 404)


""


In [10]:
df_respostas.to_csv(output_dir / "respostas_personas.csv", index=False)

## 6. Salvando resultados


In [11]:
# Salvar artefatos
personas_df.to_csv(output_dir / "hdt_personas.csv", index=False)
questionario_df.to_csv(output_dir / "questionario_7s.csv", index=False)
df_respostas.to_csv(output_dir / "hdt_respostas_7s.csv", index=False)

# Exportar no formato questionário sintético
itens = [q["item"] for q in utils.QUESTIONARIO_7S]
cols_export = ["persona_id", "sexo", "idade"] + itens
cols_export = [c for c in cols_export if c in df_respostas.columns]
df_export = df_respostas[cols_export].rename(columns={"persona_id": "Perfil"})
df_export.to_csv(f"inputs/questionario_sintetico_{len(df_respostas)}.csv", index=False)

print(f"✅ Geração concluída: {len(df_respostas)} respostas salvas.")

✅ Geração concluída: 0 respostas salvas.
